### Import Libraries 

In [1]:
import pandas as pd 
import dotenv as lead_dotenv
lead_dotenv.load_dotenv()
import requests
import json 
import os
import isodate
import re

print('done!')

done!


### Safety Check for Secret Keys:

In [2]:
# Prevent the script crash if the API isnt found or the targted channel not found
API_KEY = os.getenv('YT_API_KEY')
TARGET_CHANNEL_ID = os.getenv('CHANNEL_ID')

if not API_KEY or not TARGET_CHANNEL_ID:
    print('Missing required environment variables:')
    if not API_KEY:
        print('  - Yt_API_Key')
    if not TARGET_CHANNEL_ID:
        print('  - channel_id')
    exit()

### Get Uploads_id Function :

In [3]:
def get_uploads_id(channel_id) :
    
    url = f"https://www.googleapis.com/youtube/v3/channels?part=contentDetails&id={channel_id}&key={API_KEY}"
    print('start...')
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        
        try :
            uploads_id = data['items'][0]['contentDetails']['relatedPlaylists']['uploads']
            print(f" Success! Uploads Playlist ID found: \n{uploads_id}")
            return uploads_id
        except (KeyError, IndexError):
            print('Error: Could not parse the playlist ID from the response')
            return None
        print('end.')
    else :
        print(f"API Error : {response.status_code} : {response.text}")
print('done!')

done!


### Executing the Get Uploads_id Function using the Targeted Channel ID :

In [4]:
uploads_id = get_uploads_id(TARGET_CHANNEL_ID)

start...


 Success! Uploads Playlist ID found: 
UUcyq283he07B7_KUX07mmtA


### Get Channel Playlists id and its Statistics Funcion :

In [5]:
def get_channel_playlists(channel_id):
    
    playlists_data = []
    next_page_token = None 
    counter = 1
    
    while True:
        print(f'fitching playlists data from the page Number : {counter}')
        # Build the URL asking for snippet (title/date) and contentDetails (video count)
        url = f"https://www.googleapis.com/youtube/v3/playlists?part=snippet,contentDetails&channelId={channel_id}&maxResults=50&key={API_KEY}"
        
        # If we have a token from a previous loop, attach it to the URL
        if next_page_token:
            url += f"&pageToken={next_page_token}"
            
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            
            # Loop through the playlists in this specific page
            for item in data.get('items', []):
                snippet = item.get('snippet', {})
                content = item.get('contentDetails', {})
                
                # Build a dictionary representing one row in our future table
                row = {
                    'playlist_id': item.get('id'),
                    'playlist_name': snippet.get('title'),
                    'published_at': snippet.get('publishedAt'),
                    'total_videos': content.get('itemCount')
                }
                playlists_data.append(row)
            
            # Check if there is another page
            next_page_token = data.get('nextPageToken')
            counter += 1
            # If there is no token, we break out of the infinite while loop
            if not next_page_token:
                print(f" Finished! Found {len(playlists_data)} custom playlists.")
                break 
                
        else:
            print(f" API Error {response.status_code}: {response.text}")
            break
            
    return playlists_data

### Executing the Function Get channel playlists id and statistics :

In [6]:
channel_playlists = get_channel_playlists(TARGET_CHANNEL_ID)

if channel_playlists:
    df_playlists = pd.DataFrame(channel_playlists)
    df_playlists.to_csv('dim_playlists.csv', index=False)
    print("dim_playlists.csv saved successfully!")

fitching playlists data from the page Number : 1


fitching playlists data from the page Number : 2
 Finished! Found 51 custom playlists.
dim_playlists.csv saved successfully!


### Get all the vids Ids Function :

In [7]:
def get_all_vids_ids(playlist_id):
    
    counter = 1
    video_ids = [] 
    next_page_token = None
    
    while True:

        print(f'fitching playlists data from the page Number : {counter}')

        url = f"https://www.googleapis.com/youtube/v3/playlistItems?part=snippet&playlistId={playlist_id}&maxResults=50&key={API_KEY}"
        if next_page_token:
            url += f"&pageToken={next_page_token}"
            
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()

            for item in data.get('items', []):
                vid_id = item['snippet']['resourceId']['videoId']
                video_ids.append(vid_id)

            next_page_token = data.get('nextPageToken')
            counter += 1
            if not next_page_token:
                print(f"Finished! Extracted a total of {len(video_ids)} unique Video IDs.")
                break
        else:
            print(f"API Error {response.status_code}: {response.text}")
            break
            
    return video_ids

### Execute the Get all Vids Id function to get a list of all vids ids from Uploads :

In [8]:
all_vids_ids = get_all_vids_ids(uploads_id)

print("\nHere are the first 5 IDs we found:")
print(all_vids_ids[:5])

fitching playlists data from the page Number : 1
fitching playlists data from the page Number : 2
fitching playlists data from the page Number : 3
fitching playlists data from the page Number : 4
fitching playlists data from the page Number : 5
fitching playlists data from the page Number : 6
fitching playlists data from the page Number : 7
fitching playlists data from the page Number : 8
fitching playlists data from the page Number : 9
fitching playlists data from the page Number : 10
fitching playlists data from the page Number : 11
fitching playlists data from the page Number : 12
fitching playlists data from the page Number : 13
fitching playlists data from the page Number : 14
fitching playlists data from the page Number : 15
fitching playlists data from the page Number : 16
fitching playlists data from the page Number : 17
fitching playlists data from the page Number : 18
fitching playlists data from the page Number : 19
fitching playlists data from the page Number : 20
fitching 

### Mapping Function that Gets Vids Id for Each Playlist in the channel as One fact table with all Playlist and their vids :

In [9]:
def mapping_playlists_vids(playlists_list):
    print("\nBuilding the Playlist-to-Video Mapping Table...")
    mapping_data = []
    
    # Loop through each custom playlist we found earlier
    for playlist in playlists_list:
        p_id = playlist['playlist_id']
        next_page_token = None
        
        while True:
            # Get the items inside this specific playlist
            url = f"https://www.googleapis.com/youtube/v3/playlistItems?part=snippet&playlistId={p_id}&maxResults=50&key={API_KEY}"
            if next_page_token:
                url += f"&pageToken={next_page_token}"
                
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                
                for item in data.get('items', []):
                    v_id = item['snippet']['resourceId']['videoId']
                    # Append the pair!
                    mapping_data.append({
                        'playlist_id': p_id,
                        'video_id': v_id
                    })
                    
                next_page_token = data.get('nextPageToken')
                if not next_page_token:
                    break
            else:
                break
                
    return mapping_data

### Executing the Mapping playlist-Vids Function from channel_playlists

In [10]:
playlists_vids = mapping_playlists_vids(channel_playlists)

if playlists_vids:
    df_mapping = pd.DataFrame(playlists_vids)
    df_mapping.to_csv('fact_playlist_mapping.csv', index=False)
    print("fact_playlist_mapping.csv saved successfully!")


Building the Playlist-to-Video Mapping Table...
fact_playlist_mapping.csv saved successfully!


### Get Vids Statistics Function from the video ids list

In [11]:

def get_vids_statistics(video_id_list):
    video_data = []
    
    # Loop through the master list in chunks of 50
    for i in range(0, len(video_id_list), 50):
        
        # Slice the list to get exactly 50 IDs
        batch = video_id_list[i:i+50]

        # Join them into a single string separated by commas for the API
        ids_string = ",".join(batch)
        
        # Build the URL: 'status' is now included in the parts parameter!
        url = f"https://www.googleapis.com/youtube/v3/videos?part=snippet,contentDetails,statistics,liveStreamingDetails,status&id={ids_string}&key={API_KEY}"
        
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            
            for item in data.get('items', []):
                snippet = item.get('snippet', {})
                content = item.get('contentDetails', {})
                stats = item.get('statistics', {})
                status = item.get('status', {})
                
                # 1. Parse Duration into total seconds
                raw_duration = content.get('duration', 'PT0S')
                duration_sec = isodate.parse_duration(raw_duration).total_seconds()
                
                # 2. Find the Primary Hashtag in the description
                desc = snippet.get('description', '')
                hashtags = re.findall(r'#\w+', desc)
                primary_tag = hashtags[0] if hashtags else None
                
                # 3. Apply custom 'Content Type' Logic
                if 'liveStreamingDetails' in item:
                    content_type = 'Live Stream'
                elif duration_sec <= 60:
                    content_type = 'Short'
                else:
                    content_type = 'Standard Video'
                    
                row = {
                    'video_id': item.get('id'),
                    'title': snippet.get('title'),
                    'published_at': snippet.get('publishedAt'),
                    'duration_seconds': int(duration_sec),
                    'view_count': int(stats.get('viewCount', 0)),
                    'like_count': int(stats.get('likeCount', 0)),
                    'comment_count': int(stats.get('commentCount', 0)),
                    'category_id': snippet.get('categoryId'),
                    'tags': ", ".join(snippet.get('tags', [])), 
                    'is_hd': True if content.get('definition') == 'hd' else False,
                    'licensed_content': content.get('licensedContent', False),
                    'made_for_kids': status.get('madeForKids', False),
                    'audio_language': snippet.get('defaultAudioLanguage', 'Unknown'),
                    'has_captions': True if content.get('caption') == 'true' else False,
                    'primary_hashtag': primary_tag,
                    'content_type': content_type 
                }
                video_data.append(row)
                
            print(f"Processed batch ending at video {i + len(batch)}...")
            
        else:
            print(f"API Error on batch starting at {i}: {response.status_code}")
            
    return video_data

### Execute the Vids Statistics Function :

In [12]:
vids_statistics = get_vids_statistics(all_vids_ids)

if vids_statistics:
    df_videos = pd.DataFrame(vids_statistics)
    df_videos.to_csv('dim_videos.csv', index=False)
    print("\n dim_videos.csv saved successfully!")

Processed batch ending at video 50...
Processed batch ending at video 100...
Processed batch ending at video 150...
Processed batch ending at video 200...
Processed batch ending at video 250...
Processed batch ending at video 300...
Processed batch ending at video 350...
Processed batch ending at video 400...
Processed batch ending at video 450...
Processed batch ending at video 500...
Processed batch ending at video 550...
Processed batch ending at video 600...
Processed batch ending at video 650...
Processed batch ending at video 700...
Processed batch ending at video 750...
Processed batch ending at video 800...
Processed batch ending at video 850...
Processed batch ending at video 900...
Processed batch ending at video 950...
Processed batch ending at video 1000...
Processed batch ending at video 1050...
Processed batch ending at video 1100...
Processed batch ending at video 1150...
Processed batch ending at video 1200...
Processed batch ending at video 1250...
Processed batch endi